In [12]:
# -*- coding: utf-8 -*-

import os
import logging
import warnings
import tempfile
from pathlib import Path
import io # Importado para conversão de imagem

import numpy as np
import gradio as gr
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from PIL import Image # Importado para conversão de imagem
from scipy.io import wavfile
from scipy.signal import firwin, filtfilt, hilbert, find_peaks, welch, windows
from scipy import signal
from scipy.optimize import minimize
import math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
try:
    import ruptures as rpt
except Exception:
    rpt = None

# ==============================================================================
# SEÇÃO 1: CONFIGURAÇÕES GERAIS E DO AMBIENTE
# ==============================================================================
for proxy_var in ['http_proxy', 'https_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'all_proxy', 'ALL_PROXY']:
    os.environ.pop(proxy_var, None)
os.environ['TRANSFORMERS_OFFLINE'] = '1'
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

logger.info("✅ Configurações de ambiente carregadas")

# ==============================================================================
# SEÇÃO 2: LÓGICA DE PROCESSAMENTO DE ÁUDIO E ANÁLISE FORENSE
# ==============================================================================

def create_error_plot(message: str):
    """Cria um gráfico Plotly exibindo uma mensagem de erro."""
    fig = go.Figure()
    fig.add_annotation(
        x=0.5, y=0.5,
        text=message,
        showarrow=False,
        font=dict(size=16, color="red"),
        align="center"
    )
    fig.update_layout(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        title="Erro na Análise",
        showlegend=False,
        width=800,
        height=400
    )
    return fig

class AudioForensicsAnalyzer:
    def __init__(self):
        logger.info("✅ AudioForensicsAnalyzer inicializado")


    def calculate_enf_deviation(self, audio_path: str, fnom: float, bwenf: float):
        try:
            L = 10000 
            Fs, sut = wavfile.read(audio_path)
            if sut.ndim > 1: sut = (sut[:, 0] + sut[:, 1]) / 2
            if np.issubdtype(sut.dtype, np.integer): sut = sut.astype(np.float64) / np.iinfo(sut.dtype).max

            fs = 20 * fnom
            escala = math.gcd(int(Fs), int(fs))
            sds = signal.resample(sut, int(len(sut) * (fs/escala) / (Fs/escala)))

            hBW = bwenf / 2
            filter_coeffs = firwin(L, [2*(fnom-hBW)/fs, 2*(fnom+hBW)/fs], pass_zero=False)
            padding_size = int(1.5 * L)
            sds_padded = np.concatenate([np.zeros(padding_size), sds, np.zeros(padding_size)])
            xn = filtfilt(filter_coeffs, 1, sds_padded)
            xn = xn - np.mean(xn)
            xa = hilbert(xn)
            xa = xa[padding_size:-padding_size]
            omegaHEE = np.angle(xa[1:] * np.conj(xa[:-1]))
            enf = fs * omegaHEE / (2 * np.pi)
            enf_deviation = enf - fnom
            time_axis = np.arange(len(enf)) / fs

            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x=time_axis,
                y=enf_deviation,
                mode='lines',
                name=f'Desvio de {fnom} Hz',
                line=dict(color='blue', width=1.5),
                hovertemplate='Tempo: %{x:.3f}s<br>Desvio: %{y:.3f}Hz<extra></extra>'
            ))
            fig.add_hline(y=0, line_dash="dash", line_color="red", 
                         annotation_text="Referência (0 Hz)", annotation_position="bottom right")
            
            fig.update_layout(
                title=f'Análise de Desvio ENF - {fnom} Hz',
                xaxis_title='Tempo (segundos)',
                yaxis_title='Desvio (Hz)',
                hovermode='x',
                showlegend=True,
                legend=dict(
                    orientation="h",
                    yanchor="bottom",
                    y=1.02,
                    xanchor="right",
                    x=1
                ),
                width=900,
                height=500
            )
            fig.update_xaxes(showgrid=True, gridcolor='lightgray')
            fig.update_yaxes(showgrid=True, gridcolor='lightgray')
            
            return fig

        except Exception as e:
            logger.error(f"Erro na análise ENF: {e}")
            return create_error_plot(f"Erro na Análise ENF:\n{e}")

    def analyze_quantization_levels(self, audio_path: str, bitdepth: int, canais: int):
        """
        Análise de níveis de quantização - equivalente à função niveis() do MATLAB
        """
        try:
            # Carregar áudio mantendo informação original de canais
            Fs, sut = wavfile.read(audio_path)
            
            # Converter para float se necessário, mantendo a escala original
            if np.issubdtype(sut.dtype, np.integer):
                sut = sut.astype(np.float64)
            else:
                # Se já é float, converter para a escala de inteiros
                sut = sut * (2**(bitdepth-1))
            
            info_messages = []
            
            # Determinar qual canal usar
            if sut.ndim == 1:
                # Áudio monaural
                canal = sut
                info_messages.append('Áudio monaural')
                dc_offset = np.mean(canal)
                info_messages.append(f'Desvio DC: {dc_offset:.2f}')
                
                # Calcular histograma
                bins = np.arange(-2**(bitdepth-1), 2**(bitdepth-1)+1)
                H, _ = np.histogram(canal, bins=bins)
                
                # Plotar com Plotly
                fig = go.Figure()
                bin_centers = bins[:-1]
                fig.add_trace(go.Scatter(
                    x=bin_centers,
                    y=H,
                    mode='lines',
                    name='Histograma',
                    line=dict(color='blue', width=2),
                    hovertemplate='Nível: %{x}<br>Ocorrências: %{y}<extra></extra>'
                ))
                
                info_text = '<br>'.join(info_messages)
                fig.add_annotation(
                    x=0.02, y=0.98,
                    xref='paper', yref='paper',
                    text=info_text,
                    showarrow=False,
                    align='left',
                    bgcolor='wheat',
                    bordercolor='black',
                    borderwidth=1
                )
                
                fig.update_layout(
                    title='Histograma das amostras de áudio - todas as amostras',
                    xaxis_title='Níveis de Quantização',
                    yaxis_title='Ocorrências',
                    showlegend=False,
                    width=900,
                    height=500
                )
                fig.update_xaxes(showgrid=True, gridcolor='lightgray')
                fig.update_yaxes(showgrid=True, gridcolor='lightgray')
                
                return fig
                
            else:
                # Áudio estéreo
                if canais > 0 and canais <= sut.shape[1]:
                    # Canal específico selecionado
                    canal = sut[:, canais-1]
                    dc_offset = np.mean(canal)
                    info_messages.append(f'Canal {canais} selecionado')
                    info_messages.append(f'Desvio DC: {dc_offset:.2f}')
                    
                    # Calcular histograma
                    bins = np.arange(-2**(bitdepth-1), 2**(bitdepth-1)+1)
                    H, _ = np.histogram(canal, bins=bins)
                    
                    # Plotar com Plotly
                    fig = go.Figure()
                    bin_centers = bins[:-1]
                    fig.add_trace(go.Scatter(
                        x=bin_centers,
                        y=H,
                        mode='lines',
                        name=f'Canal {canais}',
                        line=dict(color='blue', width=2),
                        hovertemplate='Nível: %{x}<br>Ocorrências: %{y}<extra></extra>'
                    ))
                    
                    info_text = '<br>'.join(info_messages)
                    fig.add_annotation(
                        x=0.02, y=0.98,
                        xref='paper', yref='paper',
                        text=info_text,
                        showarrow=False,
                        align='left',
                        bgcolor='wheat',
                        bordercolor='black',
                        borderwidth=1
                    )
                    
                    fig.update_layout(
                        title=f'Histograma das amostras de áudio - Canal {canais}',
                        xaxis_title='Níveis de Quantização',
                        yaxis_title='Ocorrências',
                        showlegend=False,
                        width=900,
                        height=500
                    )
                    fig.update_xaxes(showgrid=True, gridcolor='lightgray')
                    fig.update_yaxes(showgrid=True, gridcolor='lightgray')
                    
                    return fig
                else:
                    # Ambos os canais (canais = 0 ou inválido)
                    info_messages.append('Áudio estéreo')
                    dc_left = np.mean(sut[:, 0])
                    dc_right = np.mean(sut[:, 1])
                    info_messages.append(f'Desvio DC do canal esquerdo: {dc_left:.2f}')
                    info_messages.append(f'Desvio DC do canal direito: {dc_right:.2f}')
                    
                    # Calcular histogramas para ambos os canais
                    bins = np.arange(-2**(bitdepth-1), 2**(bitdepth-1)+1)
                    H_left, _ = np.histogram(sut[:, 0], bins=bins)
                    H_right, _ = np.histogram(sut[:, 1], bins=bins)
                    
                    # Plotar com subplots
                    fig = make_subplots(
                        rows=2, cols=1,
                        subplot_titles=('Canal Esquerdo', 'Canal Direito'),
                        vertical_spacing=0.15
                    )
                    
                    bin_centers = bins[:-1]
                    
                    fig.add_trace(
                        go.Scatter(
                            x=bin_centers, y=H_left,
                            mode='lines', name='Canal Esquerdo',
                            line=dict(color='blue', width=2),
                            hovertemplate='Nível: %{x}<br>Ocorrências: %{y}<extra></extra>'
                        ),
                        row=1, col=1
                    )
                    
                    fig.add_trace(
                        go.Scatter(
                            x=bin_centers, y=H_right,
                            mode='lines', name='Canal Direito',
                            line=dict(color='red', width=2),
                            hovertemplate='Nível: %{x}<br>Ocorrências: %{y}<extra></extra>'
                        ),
                        row=2, col=1
                    )
                    
                    fig.update_xaxes(title_text="Níveis de Quantização", showgrid=True, gridcolor='lightgray')
                    fig.update_yaxes(title_text="Ocorrências", showgrid=True, gridcolor='lightgray')
                    
                    info_text = '<br>'.join(info_messages)
                    fig.add_annotation(
                        x=0.02, y=0.98,
                        xref='paper', yref='paper',
                        text=info_text,
                        showarrow=False,
                        align='left',
                        bgcolor='wheat',
                        bordercolor='black',
                        borderwidth=1
                    )
                    
                    fig.update_layout(
                        title='Histograma das amostras de áudio - Ambos os Canais',
                        height=700,
                        width=900,
                        showlegend=True,
                        legend=dict(
                            orientation="h",
                            yanchor="bottom",
                            y=1.02,
                            xanchor="right",
                            x=1
                        )
                    )
                    
                    return fig

        except Exception as e:
            logger.error(f"Erro na análise de níveis de quantização: {e}")
            return create_error_plot(f"Erro na Análise de Níveis:\n{e}")

    def analyze_local_dc(self, audio_path: str, dur: float):
        """
        Análise de DC local - equivalente à função localdc() do MATLAB
        dur: duração da janela em segundos
        """
        try:
            # Carregar áudio mantendo informação original
            Fs, sut = wavfile.read(audio_path)
            
            # Converter para float se necessário
            if np.issubdtype(sut.dtype, np.integer):
                # Determinar bitdepth do arquivo original
                if sut.dtype == np.int16:
                    bitdepth = 16
                elif sut.dtype == np.int32:
                    bitdepth = 32
                elif sut.dtype == np.int8:
                    bitdepth = 8
                else:
                    bitdepth = 16  # default
                sut = sut.astype(np.float64) / (2**(bitdepth-1))
            else:
                bitdepth = 16  # Assumir 16-bit para arquivos float
            
            # Separar canais
            if sut.ndim == 1:
                canal1 = sut
                canal2 = None
                num_canais = 1
            else:
                canal1 = sut[:, 0]
                canal2 = sut[:, 1] if sut.shape[1] > 1 else None
                num_canais = sut.shape[1]
            
            # Calcular número de amostras por janela
            window_samples = int(Fs * dur)
            
            # Padding para completar a última janela - EXATAMENTE como no MATLAB
            padding_needed = int(np.ceil(len(canal1) / window_samples) * window_samples) - len(canal1)
            canal1 = np.concatenate([canal1, np.zeros(padding_needed)])
            
            if canal2 is not None:
                canal2 = np.concatenate([canal2, np.zeros(padding_needed)])
            
            # Reshape em janelas - EXATAMENTE como no MATLAB
            num_windows = len(canal1) // window_samples
            canal1_windowed = canal1[:num_windows * window_samples].reshape(num_windows, window_samples)
            
            if canal2 is not None:
                canal2_windowed = canal2[:num_windows * window_samples].reshape(num_windows, window_samples)
            
            # Calcular DC médio para cada janela
            dcmean = np.zeros((num_windows, 2 if num_canais > 1 else 1))
            
            for k in range(num_windows):
                # Canal 1
                aux = canal1_windowed[k, :]
                aux_filtered = aux[np.abs(aux) < 0.95]  # Filtro como no MATLAB
                if len(aux_filtered) > 0:
                    dcmean[k, 0] = np.mean(aux_filtered) * (2**(bitdepth-1))
                
                # Canal 2 (se existir)
                if canal2 is not None:
                    aux = canal2_windowed[k, :]
                    aux_filtered = aux[np.abs(aux) < 0.95]  # Filtro como no MATLAB
                    if len(aux_filtered) > 0:
                        dcmean[k, 1] = np.mean(aux_filtered) * (2**(bitdepth-1))
            
            # Eixo temporal - centro das janelas
            time_axis = (np.arange(num_windows) + 0.5) * dur
            
            # Plotar - seguindo EXATAMENTE a lógica do MATLAB
            if num_canais == 1:
                # Apenas um canal
                fig = go.Figure()
                fig.add_trace(go.Scatter(
                    x=time_axis,
                    y=dcmean[:, 0],
                    mode='lines',
                    name='Canal Único',
                    line=dict(color='red', width=2),
                    hovertemplate='Tempo: %{x:.2f}s<br>DC: %{y:.2f}<extra></extra>'
                ))
                
                fig.update_layout(
                    title='Nível DC - Análise Local',
                    xaxis_title='Segundos',
                    yaxis_title='Nível de Quantização',
                    showlegend=False,
                    width=900,
                    height=500
                )
                fig.update_xaxes(showgrid=True, gridcolor='lightgray')
                fig.update_yaxes(showgrid=True, gridcolor='lightgray')
                
                return fig
            else:
                # Dois canais - dois gráficos como no MATLAB
                fig = make_subplots(
                    rows=2, cols=1,
                    subplot_titles=('Todos os canais', 'Equivalente monaural'),
                    vertical_spacing=0.15
                )
                
                # Primeiro gráfico - ambos os canais
                fig.add_trace(
                    go.Scatter(
                        x=time_axis, y=dcmean[:, 0],
                        mode='lines', name='Canal esquerdo',
                        line=dict(color='red', width=2),
                        hovertemplate='Tempo: %{x:.2f}s<br>DC: %{y:.2f}<extra></extra>'
                    ),
                    row=1, col=1
                )
                
                fig.add_trace(
                    go.Scatter(
                        x=time_axis, y=dcmean[:, 1],
                        mode='lines', name='Canal direito',
                        line=dict(color='red', dash='dash', width=2),
                        hovertemplate='Tempo: %{x:.2f}s<br>DC: %{y:.2f}<extra></extra>'
                    ),
                    row=1, col=1
                )
                
                # Segundo gráfico - equivalente monaural
                mono_equivalent = (dcmean[:, 0] + dcmean[:, 1]) / 2
                fig.add_trace(
                    go.Scatter(
                        x=time_axis, y=mono_equivalent,
                        mode='lines', name='Monaural',
                        line=dict(color='red', width=2),
                        hovertemplate='Tempo: %{x:.2f}s<br>DC: %{y:.2f}<extra></extra>'
                    ),
                    row=2, col=1
                )
                
                fig.update_xaxes(title_text="Segundos", showgrid=True, gridcolor='lightgray')
                fig.update_yaxes(title_text="Nível de Quantização", showgrid=True, gridcolor='lightgray')
                
                fig.update_layout(
                    title='Nível DC - Análise Local',
                    height=700,
                    width=900,
                    showlegend=True,
                    legend=dict(
                        orientation="h",
                        yanchor="bottom",
                        y=1.02,
                        xanchor="right",
                        x=1
                    )
                )
                
                return fig

        except Exception as e:
            logger.error(f"Erro na análise de DC local: {e}")
            return create_error_plot(f"Erro na Análise DC Local:\n{e}")

    def analyze_spectrogram(self, audio_path: str, fft_points: int, window_type: str, 
                           window_size_percent: float, resample_rate: float = None):
        """
        Análise de Espectrograma com controles avançados
        """
        try:
            # Carregar áudio
            audio, sr = librosa.load(audio_path, sr=resample_rate, mono=True)
            
            # Parâmetros FFT
            n_fft = 2**fft_points
            hop_length = n_fft // 4
            
            # Calcular tamanho da janela baseado na porcentagem
            window_length = int(n_fft * window_size_percent / 100.0)
            
            # Criar janela baseada no tipo
            if window_type == "hamming":
                window = np.hamming(window_length)
            elif window_type == "hanning":
                window = np.hanning(window_length)
            elif window_type == "blackman":
                window = np.blackman(window_length)
            elif window_type == "blackmanharris":
                window = windows.blackmanharris(window_length)
            elif window_type == "kaiser":
                window = np.kaiser(window_length, beta=8.6)  # beta=8.6 é similar ao blackman
            else:
                window = np.hamming(window_length)
            
            # Zero-padding se necessário
            if window_length < n_fft:
                window = np.pad(window, (0, n_fft - window_length), mode='constant')
            
            # Calcular espectrograma usando STFT
            stft = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length, window=window)
            magnitude = np.abs(stft)
            magnitude_db = librosa.amplitude_to_db(magnitude, ref=np.max)
            
            # Eixos de tempo e frequência
            times = librosa.frames_to_time(np.arange(magnitude.shape[1]), sr=sr, hop_length=hop_length)
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
            
            # Criar espectrograma com Plotly
            fig = go.Figure(data=go.Heatmap(
                z=magnitude_db,
                x=times,
                y=freqs,
                colorscale='Viridis',
                hovertemplate='Tempo: %{x:.3f}s<br>Freq: %{y:.1f}Hz<br>Magnitude: %{z:.1f}dB<extra></extra>',
                colorbar=dict(title="Magnitude (dB)")
            ))
            
            fig.update_layout(
                title=f'Espectrograma - {window_type.title()} Window ({window_size_percent}%)',
                xaxis_title='Tempo (segundos)',
                yaxis_title='Frequência (Hz)',
                width=1000,
                height=600
            )
            
            # Configurar eixos
            fig.update_xaxes(showgrid=True, gridcolor='lightgray')
            fig.update_yaxes(showgrid=True, gridcolor='lightgray')
            
            # Adicionar informações no gráfico
            info_text = f"FFT: {n_fft} pontos | Janela: {window_type} ({window_length} amostras) | SR: {sr} Hz"
            fig.add_annotation(
                x=0.02, y=0.98,
                xref='paper', yref='paper',
                text=info_text,
                showarrow=False,
                align='left',
                bgcolor='rgba(255,255,255,0.8)',
                bordercolor='black',
                borderwidth=1
            )
            
            return fig
            
        except Exception as e:
            logger.error(f"Erro na análise de espectrograma: {e}")
            return create_error_plot(f"Erro no Espectrograma:\n{e}")

    def analyze_ltas(self, audio_path: str, pts: int, canais: int, resample_rate: float = None):
        """
        Análise LTAS - equivalente à função LTAS() do MATLAB
        pts: número de pontos para pwelch
        canais: canal selecionado (0=monaural equivalente, 1=esquerdo, 2=direito)
        resample_rate: taxa de reamostragem opcional
        """
        try:
            # Carregar áudio com reamostragem opcional
            if resample_rate is not None:
                audio, Fs = librosa.load(audio_path, sr=resample_rate, mono=False)
                if audio.ndim == 1:
                    sut = audio
                else:
                    sut = audio.T  # Transpor para formato (samples, channels)
            else:
                Fs, sut = wavfile.read(audio_path)
            
            # Converter para float se necessário
            if np.issubdtype(sut.dtype, np.integer):
                sut = sut.astype(np.float64) / np.iinfo(sut.dtype).max
            
            # Selecionar canal conforme lógica do MATLAB
            if sut.ndim == 1:
                canal = sut
                canal_info = "Áudio monaural"
            elif canais > 0 and canais <= sut.shape[1]:
                canal = sut[:, canais-1]
                canal_info = f"Canal {canais}"
            else:
                # Equivalente monaural (como no MATLAB)
                if sut.shape[1] >= 2:
                    canal = sut[:, 0]/2 + sut[:, 1]/2
                else:
                    canal = sut[:, 0]
                canal_info = "Equivalente monaural"
            
            # Calcular PSD usando método de Welch - EXATAMENTE como no MATLAB
            ff, psd = welch(canal, fs=Fs, nperseg=pts, return_onesided=True, scaling='density')
            
            # Normalizar PSD - EXATAMENTE como no MATLAB: psd=psd/norm(psd)
            psd = psd / np.linalg.norm(psd)
            
            # PSD ordenado em ordem decrescente - como no MATLAB
            spsd = np.sort(psd)[::-1]
            
            # Calcular derivada do LTAS ordenado (diferencial entre amostras)
            spsd_derivative = np.diff(spsd)
            ff_derivative = ff[1:]  # Eixo de frequência para derivada (uma amostra a menos)
            
            # Gráfico 1: LTAS Normal
            psd_db = 10*np.log10(psd + 1e-10)
            fig_normal = go.Figure()
            fig_normal.add_trace(
                go.Scatter(
                    x=ff, y=psd_db,
                    mode='lines', name='LTAS Normal',
                    line=dict(color='blue', width=1.5),
                    hovertemplate='Freq: %{x:.1f}Hz<br>Mag: %{y:.1f}dB<extra></extra>'
                )
            )
            fig_normal.update_layout(
                title=f'LTAS Normal - Método de Welch - {canal_info}',
                xaxis_title='Frequência (Hz)',
                yaxis_title='Magnitude (dB)',
                width=600,
                height=400
            )
            fig_normal.update_xaxes(showgrid=True, gridcolor='lightgray')
            fig_normal.update_yaxes(showgrid=True, gridcolor='lightgray')
            
            # Gráfico 2: LTAS com compensação 6dB/oitava
            ff_comp = np.where(ff > 0, ff, 1e-10)
            psd_compensated = psd_db + 20*np.log10(ff_comp)
            fig_6db = go.Figure()
            fig_6db.add_trace(
                go.Scatter(
                    x=ff, y=psd_compensated,
                    mode='lines', name='LTAS 6dB/oitava',
                    line=dict(color='orange', width=1.5),
                    hovertemplate='Freq: %{x:.1f}Hz<br>Mag: %{y:.1f}dB<extra></extra>'
                )
            )
            fig_6db.update_layout(
                title=f'LTAS 6dB/oitava - Método de Welch - {canal_info}',
                xaxis_title='Frequência (Hz)',
                yaxis_title='Magnitude (dB)',
                width=600,
                height=400
            )
            fig_6db.update_xaxes(showgrid=True, gridcolor='lightgray')
            fig_6db.update_yaxes(showgrid=True, gridcolor='lightgray')
            
            # Gráfico 3: LTAS Ordenado
            spsd_db = 10*np.log10(spsd + 1e-10)
            fig_sorted = go.Figure()
            fig_sorted.add_trace(
                go.Scatter(
                    x=ff, y=spsd_db,
                    mode='lines', name='LTAS Ordenado',
                    line=dict(color='green', width=1.5),
                    hovertemplate='Freq: %{x:.1f}Hz<br>Mag: %{y:.1f}dB<extra></extra>'
                )
            )
            fig_sorted.update_layout(
                title=f'LTAS Ordenado - Método de Welch - {canal_info}',
                xaxis_title='Frequência (Hz)',
                yaxis_title='Magnitude (dB)',
                width=600,
                height=400
            )
            fig_sorted.update_xaxes(showgrid=True, gridcolor='lightgray')
            fig_sorted.update_yaxes(showgrid=True, gridcolor='lightgray')
            
            # Gráfico 4: Derivada do LTAS Ordenado
            # Calcular derivada do LTAS ordenado em dB (diferencial entre amostras)
            spsd_derivative = np.diff(spsd_db)
            ff_derivative = ff[1:]  # Eixo de frequência para derivada (uma amostra a menos)
            fig_derivative = go.Figure()
            fig_derivative.add_trace(
                go.Scatter(
                    x=ff_derivative, y=spsd_derivative,
                    mode='lines', name='Derivada LTAS Ordenado',
                    line=dict(color='red', width=1.5),
                    hovertemplate='Freq: %{x:.1f}Hz<br>Derivada: %{y:.6f}<extra></extra>'
                )
            )
            fig_derivative.update_layout(
                title=f'Derivada LTAS Ordenado - {canal_info}',
                xaxis_title='Frequência (Hz)',
                yaxis_title='Diferencial (dPSD/df)',
                width=600,
                height=400
            )
            fig_derivative.update_xaxes(showgrid=True, gridcolor='lightgray')
            fig_derivative.update_yaxes(showgrid=True, gridcolor='lightgray')
            
            return fig_normal, fig_6db, fig_sorted, fig_derivative

        except Exception as e:
            logger.error(f"Erro na análise LTAS: {e}")
            error_plot = create_error_plot(f"Erro na Análise LTAS:\n{e}")
            return error_plot, error_plot, error_plot, error_plot

# ==============================================================================
# SEÇÃO 3: LÓGICA DE PREDIÇÃO E CARREGAMENTO
# ==============================================================================

def load_audio_analysis_models():
    models = {}
    logger.info("🚀 Carregando analisador de áudio forense...")
    try:
        analyzer = AudioForensicsAnalyzer()
        models["forensics_analyzer"] = analyzer
        logger.info("✅ Analisador de áudio forense carregado com sucesso!")
    except Exception as e: logger.error(f"Falha ao carregar analisador: {e}")
    return models

# Sistema de cores para sobreposição
COLOR_PALETTE = [
    ['blue', 'red'],                    # Primeiro áudio 
    ['green', 'orange'],                # Segundo áudio
    ['purple', 'brown'],                # Terceiro áudio
    ['pink', 'gray'],                   # Quarto áudio
    ['cyan', 'magenta'],                # Quinto áudio
    ['lime', 'navy'],                   # Sexto áudio
]

def get_colors_for_audio(audio_count):
    """Retorna as cores para o áudio baseado no contador"""
    return COLOR_PALETTE[audio_count % len(COLOR_PALETTE)]

def process_stereo_differential(audio_input, stereo_diff_enabled):
    """
    Processa resíduo diferencial estéreo se habilitado e áudio for estéreo
    Mantém os valores dentro dos limites de quantização originais
    """
    if audio_input is None or not stereo_diff_enabled:
        return audio_input
    
    sample_rate, data = audio_input
    
    # Verificar se é estéreo
    if data.ndim == 1:
        # Áudio monaural - retornar sem modificação
        return audio_input
    
    if data.shape[1] < 2:
        # Apenas um canal - retornar sem modificação
        return audio_input
    
    # Processar resíduo diferencial estéreo
    left_channel = data[:, 0]
    right_channel = data[:, 1]
    
    # Resíduo diferencial: (right - left) / 2
    # IMPORTANTE: Fazer o cálculo no tipo original para evitar overflow
    if np.issubdtype(data.dtype, np.integer):
        # Para dados inteiros, fazer o cálculo em int64 para evitar overflow
        left_int64 = left_channel.astype(np.int64)
        right_int64 = right_channel.astype(np.int64)
        differential_residue = (right_int64 - left_int64) // 2  # Divisão inteira
        
        # Determinar os limites baseado no tipo de dados original
        if data.dtype == np.int16:
            min_val, max_val = -32768, 32767
        elif data.dtype == np.int32:
            min_val, max_val = -2147483648, 2147483647
        elif data.dtype == np.int8:
            min_val, max_val = -128, 127
        else:
            # Para outros tipos inteiros, usar limites genéricos
            min_val, max_val = np.iinfo(data.dtype).min, np.iinfo(data.dtype).max
        
        # Clamp os valores para dentro dos limites
        differential_residue = np.clip(differential_residue, min_val, max_val)
        
        # Converter para o tipo original
        differential_residue = differential_residue.astype(data.dtype)
    else:
        # Para dados float, fazer cálculo normal
        differential_residue = (right_channel - left_channel) / 2.0
        # Manter entre -1.0 e 1.0
        differential_residue = np.clip(differential_residue, -1.0, 1.0)
    
    # Converter para monaural
    processed_data = differential_residue.reshape(-1, 1)
    
    return sample_rate, processed_data

def predict_audio_forensics(audio_input, analysis_models, fnom, bwenf, stereo_diff_enabled=False):
    if audio_input is None: return None
    
    # Processar resíduo diferencial estéreo se habilitado
    processed_audio = process_stereo_differential(audio_input, stereo_diff_enabled)
    
    temp_path = None
    try:
        sample_rate, data = processed_audio
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmpfile:
            temp_path = tmpfile.name
            if np.issubdtype(data.dtype, np.integer):
                data = data.astype(np.float32) / np.iinfo(data.dtype).max
            sf.write(temp_path, data, sample_rate)
        
        analyzer = analysis_models["forensics_analyzer"]
        
        enf_fig = analyzer.calculate_enf_deviation(temp_path, fnom=fnom, bwenf=bwenf)
        
        return enf_fig
            
    except Exception as e:
        logger.error(f"Erro na análise: {e}", exc_info=True)
        error_plot = create_error_plot(f"Erro na análise: {e}")
        return error_plot
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

def predict_enf_with_overlay(audio_input, models, fnom, bwenf, stereo_diff_enabled, retain_data, previous_fig, audio_counter):
    if audio_input is None: 
        return previous_fig if retain_data else None, audio_counter
    
    # Gerar nova análise
    new_fig = predict_audio_forensics(audio_input, models, fnom, bwenf, stereo_diff_enabled)
    
    if not retain_data or previous_fig is None:
        # Primeiro áudio ou não reter dados
        audio_counter = 0
        return new_fig, audio_counter + 1
    
    # Adicionar novo trace ao gráfico existente
    colors = get_colors_for_audio(audio_counter)
    
    # Combinar gráficos
    combined_fig = go.Figure(previous_fig)
    
    # Extrair dados do novo gráfico
    new_trace = new_fig.data[0]
    new_trace.line.color = colors[0]
    new_trace.name = f'Áudio {audio_counter + 1} - Desvio de {fnom} Hz'
    
    combined_fig.add_trace(new_trace)
    combined_fig.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return combined_fig, audio_counter + 1


def predict_levels_with_overlay(audio_input, models, bitdepth, canais, stereo_diff_enabled, retain_data, previous_fig, audio_counter):
    if audio_input is None:
        return previous_fig if retain_data else None, audio_counter
    
    new_fig = predict_quantization_analysis(audio_input, models, bitdepth, canais, stereo_diff_enabled)
    
    if not retain_data or previous_fig is None:
        audio_counter = 0
        return new_fig, audio_counter + 1
    
    colors = get_colors_for_audio(audio_counter)
    combined_fig = go.Figure(previous_fig)
    
    for i, trace in enumerate(new_fig.data):
        trace.line.color = colors[i % 2]
        trace.name = f'Áudio {audio_counter + 1} - {trace.name}'
        combined_fig.add_trace(trace)
    
    combined_fig.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    return combined_fig, audio_counter + 1

def predict_dc_with_overlay(audio_input, models, window_duration_s, stereo_diff_enabled, retain_data, previous_fig, audio_counter):
    if audio_input is None:
        return previous_fig if retain_data else None, audio_counter
    
    new_fig = predict_local_dc_analysis(audio_input, models, window_duration_s, stereo_diff_enabled)
    
    if not retain_data or previous_fig is None:
        audio_counter = 0
        return new_fig, audio_counter + 1
    
    colors = get_colors_for_audio(audio_counter)
    combined_fig = go.Figure(previous_fig)
    
    for i, trace in enumerate(new_fig.data):
        trace.line.color = colors[i % 2]
        trace.name = f'Áudio {audio_counter + 1} - {trace.name}'
        combined_fig.add_trace(trace)
    
    combined_fig.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    return combined_fig, audio_counter + 1

def predict_ltas_with_overlay(audio_input, models, fft_points, canais, resample_rate, stereo_diff_enabled, retain_data, 
                            prev_normal, prev_6db, prev_sorted, prev_derivative, audio_counter):
    if audio_input is None:
        return (prev_normal if retain_data else None, 
                prev_6db if retain_data else None,
                prev_sorted if retain_data else None, 
                prev_derivative if retain_data else None, audio_counter)
    
    new_normal, new_6db, new_sorted, new_derivative = predict_ltas_analysis(audio_input, models, fft_points, canais, resample_rate, stereo_diff_enabled)
    
    if not retain_data or prev_normal is None:
        audio_counter = 0
        return new_normal, new_6db, new_sorted, new_derivative, audio_counter + 1
    
    colors = get_colors_for_audio(audio_counter)
    
    # Combinar cada gráfico
    combined_normal = go.Figure(prev_normal)
    trace = new_normal.data[0]
    trace.line.color = colors[0]
    trace.name = f'Áudio {audio_counter + 1} - LTAS Normal'
    combined_normal.add_trace(trace)
    combined_normal.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    combined_6db = go.Figure(prev_6db)
    trace = new_6db.data[0]
    trace.line.color = colors[0]
    trace.name = f'Áudio {audio_counter + 1} - LTAS 6dB/oitava'
    combined_6db.add_trace(trace)
    combined_6db.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    combined_sorted = go.Figure(prev_sorted)
    trace = new_sorted.data[0]
    trace.line.color = colors[0]
    trace.name = f'Áudio {audio_counter + 1} - LTAS Ordenado'
    combined_sorted.add_trace(trace)
    combined_sorted.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    combined_derivative = go.Figure(prev_derivative)
    trace = new_derivative.data[0]
    trace.line.color = colors[0]
    trace.name = f'Áudio {audio_counter + 1} - Derivada'
    combined_derivative.add_trace(trace)
    combined_derivative.update_layout(
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    return combined_normal, combined_6db, combined_sorted, combined_derivative, audio_counter + 1

def predict_quantization_analysis(audio_input, analysis_models, bitdepth, canais, stereo_diff_enabled=False):
    if audio_input is None: return None
    
    # Processar resíduo diferencial estéreo se habilitado
    processed_audio = process_stereo_differential(audio_input, stereo_diff_enabled)
    
    temp_path = None
    try:
        sample_rate, data = processed_audio
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmpfile:
            temp_path = tmpfile.name
            # Preservar dados originais sem normalização para análise de quantização
            sf.write(temp_path, data, sample_rate)
        
        analyzer = analysis_models["forensics_analyzer"]
        result = analyzer.analyze_quantization_levels(temp_path, bitdepth=int(bitdepth), canais=int(canais))
        return result
            
    except Exception as e:
        logger.error(f"Erro na análise de níveis: {e}", exc_info=True)
        return create_error_plot(f"Erro na análise de níveis: {e}")
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

def predict_local_dc_analysis(audio_input, analysis_models, window_duration_s, stereo_diff_enabled=False):
    if audio_input is None: return None
    
    # Processar resíduo diferencial estéreo se habilitado
    processed_audio = process_stereo_differential(audio_input, stereo_diff_enabled)
    
    temp_path = None
    try:
        sample_rate, data = processed_audio
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmpfile:
            temp_path = tmpfile.name
            # Preservar dados originais sem normalização
            sf.write(temp_path, data, sample_rate)
        
        analyzer = analysis_models["forensics_analyzer"]
        result = analyzer.analyze_local_dc(temp_path, dur=window_duration_s)
        return result
            
    except Exception as e:
        logger.error(f"Erro na análise DC local: {e}", exc_info=True)
        return create_error_plot(f"Erro na análise DC local: {e}")
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

def predict_spectrogram_analysis(audio_input, analysis_models, fft_points, window_type, window_size_percent, resample_rate, stereo_diff_enabled=False):
    if audio_input is None: return None
    
    # Processar resíduo diferencial estéreo se habilitado
    processed_audio = process_stereo_differential(audio_input, stereo_diff_enabled)
    
    temp_path = None
    try:
        sample_rate, data = processed_audio
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmpfile:
            temp_path = tmpfile.name
            if np.issubdtype(data.dtype, np.integer):
                data = data.astype(np.float32) / np.iinfo(data.dtype).max
            sf.write(temp_path, data, sample_rate)
        
        analyzer = analysis_models["forensics_analyzer"]
        result = analyzer.analyze_spectrogram(
            temp_path, 
            fft_points=int(fft_points), 
            window_type=window_type, 
            window_size_percent=float(window_size_percent),
            resample_rate=float(resample_rate) if resample_rate else None
        )
        return result
            
    except Exception as e:
        logger.error(f"Erro na análise de espectrograma: {e}", exc_info=True)
        return create_error_plot(f"Erro na análise de espectrograma: {e}")
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

def predict_spectrogram_with_overlay(audio_input, models, fft_points, window_type, window_size_percent, resample_rate, stereo_diff_enabled, retain_data, previous_fig, audio_counter):
    if audio_input is None:
        return previous_fig if retain_data else None, audio_counter
    
    new_fig = predict_spectrogram_analysis(audio_input, models, fft_points, window_type, window_size_percent, resample_rate, stereo_diff_enabled)
    
    if not retain_data or previous_fig is None:
        audio_counter = 0
        return new_fig, audio_counter + 1
    
    colors = get_colors_for_audio(audio_counter)
    combined_fig = go.Figure(previous_fig)
    
    # Para espectrograma, vamos adicionar uma nova camada de heatmap
    # (Plotly não permite sobreposição direta de heatmaps, então vamos criar um novo)
    combined_fig = new_fig  # Por simplicidade, vamos substituir o anterior
    combined_fig.update_layout(
        title=f'Espectrograma - {window_type.title()} Window ({window_size_percent}%) - Áudio {audio_counter + 1}'
    )
    
    return combined_fig, audio_counter + 1

def predict_ltas_analysis(audio_input, analysis_models, fft_points, canais, resample_rate, stereo_diff_enabled=False):
    if audio_input is None: return None, None, None, None
    
    # Processar resíduo diferencial estéreo se habilitado
    processed_audio = process_stereo_differential(audio_input, stereo_diff_enabled)
    
    temp_path = None
    try:
        sample_rate, data = processed_audio
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmpfile:
            temp_path = tmpfile.name
            if np.issubdtype(data.dtype, np.integer):
                data = data.astype(np.float32) / np.iinfo(data.dtype).max
            sf.write(temp_path, data, sample_rate)
        
        analyzer = analysis_models["forensics_analyzer"]
        fig_normal, fig_6db, fig_sorted, fig_derivative = analyzer.analyze_ltas(
            temp_path, 
            pts=fft_points, 
            canais=int(canais),
            resample_rate=float(resample_rate) if resample_rate else None
        )
        return fig_normal, fig_6db, fig_sorted, fig_derivative
            
    except Exception as e:
        logger.error(f"Erro na análise LTAS: {e}", exc_info=True)
        error_plot = create_error_plot(f"Erro na análise LTAS: {e}")
        return error_plot, error_plot, error_plot, error_plot
    finally:
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

# ==============================================================================
# SEÇÃO 4: INTERFACE GRADIO
# ==============================================================================

if __name__ == "__main__":
    analysis_models = load_audio_analysis_models()
    
    if not analysis_models:
        logger.error("❌ ERRO CRÍTICO: Falha ao carregar o analisador.")
    else:
        logger.info("✅ Analisador carregado. Iniciando Gradio...")
        with gr.Blocks(theme=gr.themes.Soft()) as demo:
            gr.Markdown("# 🔍 Análise Forense de Áudio Interativa")
            gr.Markdown("Faça o upload de um arquivo de áudio para análise forense completa com **gráficos interativos** (zoom, pan, hover).")
            
            with gr.Row():
                with gr.Column(scale=1):
                    audio_input = gr.Audio(label="Áudio de Entrada", type='numpy')
            
            # Controle global para resíduo diferencial estéreo
            with gr.Row():
                with gr.Column(scale=1):
                    stereo_diff_checkbox = gr.Checkbox(
                        label="🔄 Gerar Resíduo Diferencial Estéreo", 
                        value=False,
                        info="Inverte amplitude do canal esquerdo, soma ao direito e divide por 2. Ativo apenas para áudio estéreo."
                    )
            
            with gr.Tabs():
                # ABA 1: ESPECTROGRAMA
                with gr.TabItem("📊 Espectrograma"):
                    with gr.Accordion("Parâmetros do Espectrograma", open=True):
                        with gr.Row():
                            fft_points_input = gr.Slider(label="Pontos FFT (2^N)", minimum=8, maximum=16, value=12, step=1, info="Número de pontos da FFT como 2^N")
                            window_type_input = gr.Dropdown(label="Tipo de Janela", choices=["hamming", "hanning", "blackman", "blackmanharris", "kaiser"], value="hamming", info="Tipo de janela para análise")
                        with gr.Row():
                            window_size_input = gr.Slider(label="Tamanho da Janela (%)", minimum=10, maximum=100, value=75, step=5, info="Percentual do tamanho da janela em relação aos pontos FFT")
                            resample_rate_input = gr.Number(label="Taxa de Reamostragem (Hz)", value=None, info="Deixe vazio para manter taxa original")
                        with gr.Row():
                            analyze_spectrogram_button = gr.Button("🚀 Gerar Espectrograma", variant="primary")
                            retain_spectrogram_checkbox = gr.Checkbox(label="📌 Reter dados para comparação", value=False)
                            clear_spectrogram_button = gr.Button("🗑️ Limpar", variant="secondary")
                    
                    spectrogram_output = gr.Plot(label="Espectrograma - Interativo")
                    
                    gr.Markdown("""
                    **🔍 Navegação Interativa:** Zoom, pan, hover com valores exatos
                    **📌 Comparação:** Marque "Reter dados" para sobrepor áudios diferentes
                    
                    **Como Interpretar:** 
                    - **Eixo X:** Tempo (segundos)
                    - **Eixo Y:** Frequência (Hz)
                    - **Cor:** Magnitude (dB) - cores quentes = mais energia
                    
                    **Parâmetros:**
                    - **Pontos FFT:** Maior = melhor resolução frequencial, menor = melhor resolução temporal
                    - **Janela:** Hamming (padrão), Hanning (suave), Blackman (menos vazamento), Kaiser (ajustável)
                    - **Tamanho da Janela:** 100% = sem zero-padding, 50% = janela ocupa metade dos pontos FFT
                    - **Reamostragem:** Reduz taxa de amostragem para análise de baixas frequências
                    """)
                
                # ABA 2: ANÁLISE ENF
                with gr.TabItem("🌊 Análise ENF"):
                    with gr.Accordion("Parâmetros ENF", open=True):
                        with gr.Row():
                            enf_freq_input = gr.Number(label="Frequência ENF Nominal (fnom)", value=60.0, info="Insira a frequência nominal em Hz. Comum: 60 (Américas) ou 50 (Brasil/Europa).")
                            enf_bw_input = gr.Slider(label="Largura de Banda ENF (BWenf)", minimum=0.1, maximum=2.0, value=0.8, step=0.1, info="Largura do filtro para isolar o sinal ENF.")
                        with gr.Row():
                            analyze_enf_button = gr.Button("🚀 Analisar ENF", variant="primary")
                            retain_enf_checkbox = gr.Checkbox(label="📌 Reter dados para comparação", value=False)
                            clear_enf_button = gr.Button("🗑️ Limpar", variant="secondary")
                    
                    enf_output = gr.Plot(label="Análise de Desvio ENF - Interativo")
                    
                    gr.Markdown("""
                    **🔍 Navegação Interativa:**
                    - **Zoom:** Selecione área ou use roda do mouse
                    - **Pan:** Clique e arraste 
                    - **Reset:** Clique duplo
                    - **Hover:** Valores exatos ao passar o mouse
                    - **📌 Comparação:** Marque "Reter dados" para sobrepor áudios diferentes
                    
                    **Como Interpretar:** Procure por **descontinuidades ou saltos abruptos** no desvio da frequência ENF.
                    """)
                
                # ABA 2: ANÁLISE DE NÍVEIS DE QUANTIZAÇÃO
                with gr.TabItem("📊 Análise de Níveis"):
                    with gr.Accordion("Parâmetros de Quantização", open=True):
                        with gr.Row():
                            bitdepth_input = gr.Number(label="Bits por Amostra", value=16, info="Profundidade de bits do áudio (8, 16, 24, 32)")
                            canais_quant_input = gr.Number(label="Canal (0=ambos, 1=esquerdo, 2=direito)", value=0, info="Selecione o canal para análise (como no MATLAB)")
                        with gr.Row():
                            analyze_levels_button = gr.Button("🚀 Analisar Níveis", variant="primary")
                            retain_levels_checkbox = gr.Checkbox(label="📌 Reter dados para comparação", value=False)
                            clear_levels_button = gr.Button("🗑️ Limpar", variant="secondary")
                    
                    levels_output = gr.Plot(label="Histograma de Níveis de Quantização - Interativo")
                    
                    gr.Markdown("""
                    **🔍 Navegação Interativa:** Zoom para investigar regiões específicas do histograma
                    **📌 Comparação:** Marque "Reter dados" para sobrepor áudios diferentes
                    
                    **Como Interpretar:** O histograma mostra a distribuição dos níveis de quantização.
                    Gaps ou padrões incomuns podem indicar processamentos aplicados ao áudio.
                    """)
                
                # ABA 3: ANÁLISE DC LOCAL
                with gr.TabItem("⚡ Análise DC Local"):
                    with gr.Accordion("Parâmetros DC Local", open=True):
                        with gr.Row():
                            window_duration_input = gr.Number(label="Duração da Janela (segundos)", value=5.0, info="Tamanho da janela de análise em segundos (como dur no MATLAB)")
                        with gr.Row():
                            analyze_dc_button = gr.Button("🚀 Analisar DC Local", variant="primary")
                            retain_dc_checkbox = gr.Checkbox(label="📌 Reter dados para comparação", value=False)
                            clear_dc_button = gr.Button("🗑️ Limpar", variant="secondary")
                    
                    dc_output = gr.Plot(label="Análise de Nível DC Local - Interativo")
                    
                    gr.Markdown("""
                    **🔍 Navegação Interativa:** Zoom temporal para investigar variações suspeitas
                    **📌 Comparação:** Marque "Reter dados" para sobrepor áudios diferentes
                    
                    **Como Interpretar:** Variações abruptas no nível DC podem indicar edições,
                    cortes ou emendas no material de áudio.
                    """)
                
                # ABA 4: ANÁLISES LTAS
                with gr.TabItem("📈 Análises LTAS"):
                    with gr.Accordion("Parâmetros LTAS", open=True):
                        with gr.Row():
                            fft_n_input = gr.Slider(label="N para FFT (2^N pontos)", minimum=8, maximum=16, value=12, step=1, info="Número de pontos da FFT como 2^N (pts no MATLAB)")
                            canais_ltas_input = gr.Number(label="Canal (0=monaural equiv., 1=esquerdo, 2=direito)", value=0, info="Selecione o canal (como canais no MATLAB)")
                        with gr.Row():
                            resample_ltas_input = gr.Number(label="Taxa de Reamostragem (Hz)", value=None, info="Deixe vazio para manter taxa original")
                        with gr.Row():
                            analyze_ltas_button = gr.Button("🚀 Analisar LTAS", variant="primary")
                            retain_ltas_checkbox = gr.Checkbox(label="📌 Reter dados para comparação", value=False)
                            clear_ltas_button = gr.Button("🗑️ Limpar", variant="secondary")
                    
                    with gr.Row():
                        ltas_normal_output = gr.Plot(label="LTAS Normal - Interativo")
                        ltas_6db_output = gr.Plot(label="LTAS 6dB/oitava - Interativo")
                    with gr.Row():
                        ltas_sorted_output = gr.Plot(label="LTAS Ordenado - Interativo")
                        ltas_derivative_output = gr.Plot(label="Derivada LTAS Ordenado - Interativo")
                    
                    gr.Markdown("""
                    **🔍 Navegação Interativa:** Zoom espectral para investigar frequências específicas
                    **📌 Comparação:** Marque "Reter dados" para sobrepor áudios diferentes
                    
                    **Como Interpretar:** 
                    - **LTAS Normal:** Espectro médio de longo termo usando método de Welch
                    - **LTAS 6dB/oitava:** Compensação espectral para análise de ruído rosa
                    - **LTAS Ordenado:** Espectro ordenado por magnitude decrescente
                    - **Derivada LTAS:** Diferencial entre amostras do espectro ordenado
                    
                    **Reamostragem:** Reduz taxa de amostragem para análise de baixas frequências ou processamento mais rápido
                    """)
            
            # Estados para manter os gráficos anteriores e contadores
            spectrogram_state = gr.State(None)
            spectrogram_counter = gr.State(0)
            enf_state = gr.State(None)
            enf_counter = gr.State(0)
            levels_state = gr.State(None)
            levels_counter = gr.State(0)
            dc_state = gr.State(None)
            dc_counter = gr.State(0)
            ltas_normal_state = gr.State(None)
            ltas_6db_state = gr.State(None)
            ltas_sorted_state = gr.State(None)
            ltas_derivative_state = gr.State(None)
            ltas_counter = gr.State(0)
            
            # CONFIGURAÇÃO DOS EVENTOS DOS BOTÕES
            
            # Espectrograma com sobreposição
            def spectrogram_analysis_wrapper(audio, fft_points, window_type, window_size_percent, resample_rate, stereo_diff, retain, prev_fig, counter):
                if audio is None:
                    return prev_fig if retain else None, counter, prev_fig if retain else None
                new_fig, new_counter = predict_spectrogram_with_overlay(audio, analysis_models, fft_points, window_type, window_size_percent, resample_rate, stereo_diff, retain, prev_fig, counter)
                return new_fig, new_counter, new_fig
            
            analyze_spectrogram_button.click(
                fn=spectrogram_analysis_wrapper,
                inputs=[audio_input, fft_points_input, window_type_input, window_size_input, resample_rate_input,
                       stereo_diff_checkbox, retain_spectrogram_checkbox, spectrogram_state, spectrogram_counter],
                outputs=[spectrogram_output, spectrogram_counter, spectrogram_state]
            )
            
            clear_spectrogram_button.click(
                fn=lambda: (None, 0, None),
                outputs=[spectrogram_output, spectrogram_counter, spectrogram_state]
            )
            
            # ENF com sobreposição
            def enf_analysis_wrapper(audio, fnom, bwenf, stereo_diff, retain, prev_fig, counter):
                if audio is None:
                    return prev_fig if retain else None, counter, prev_fig if retain else None
                new_fig, new_counter = predict_enf_with_overlay(audio, analysis_models, fnom, bwenf, stereo_diff, retain, prev_fig, counter)
                return new_fig, new_counter, new_fig  # O último é para atualizar o state
            
            analyze_enf_button.click(
                fn=enf_analysis_wrapper,
                inputs=[audio_input, enf_freq_input, enf_bw_input, stereo_diff_checkbox,
                       retain_enf_checkbox, enf_state, enf_counter],
                outputs=[enf_output, enf_counter, enf_state]
            )
            
            clear_enf_button.click(
                fn=lambda: (None, 0, None),
                outputs=[enf_output, enf_counter, enf_state]
            )
            
            # Níveis com sobreposição
            def levels_analysis_wrapper(audio, bitdepth, canais, stereo_diff, retain, prev_fig, counter):
                if audio is None:
                    return prev_fig if retain else None, counter, prev_fig if retain else None
                new_fig, new_counter = predict_levels_with_overlay(audio, analysis_models, bitdepth, canais, stereo_diff, retain, prev_fig, counter)
                return new_fig, new_counter, new_fig
            
            analyze_levels_button.click(
                fn=levels_analysis_wrapper,
                inputs=[audio_input, bitdepth_input, canais_quant_input, stereo_diff_checkbox,
                       retain_levels_checkbox, levels_state, levels_counter],
                outputs=[levels_output, levels_counter, levels_state]
            )
            
            clear_levels_button.click(
                fn=lambda: (None, 0, None),
                outputs=[levels_output, levels_counter, levels_state]
            )
            
            # DC Local com sobreposição
            def dc_analysis_wrapper(audio, window_dur, stereo_diff, retain, prev_fig, counter):
                if audio is None:
                    return prev_fig if retain else None, counter, prev_fig if retain else None
                new_fig, new_counter = predict_dc_with_overlay(audio, analysis_models, window_dur, stereo_diff, retain, prev_fig, counter)
                return new_fig, new_counter, new_fig
            
            analyze_dc_button.click(
                fn=dc_analysis_wrapper,
                inputs=[audio_input, window_duration_input, stereo_diff_checkbox,
                       retain_dc_checkbox, dc_state, dc_counter],
                outputs=[dc_output, dc_counter, dc_state]
            )
            
            clear_dc_button.click(
                fn=lambda: (None, 0, None),
                outputs=[dc_output, dc_counter, dc_state]
            )
            
            # LTAS com sobreposição
            def ltas_analysis_wrapper(audio, n_slider, canais, resample_rate, stereo_diff, retain, prev_normal, prev_6db, 
                                    prev_sorted, prev_deriv, counter):
                if audio is None:
                    return (prev_normal if retain else None, prev_6db if retain else None,
                           prev_sorted if retain else None, prev_deriv if retain else None, counter,
                           prev_normal if retain else None, prev_6db if retain else None,
                           prev_sorted if retain else None, prev_deriv if retain else None)
                fft_points = 2**int(n_slider)
                n1, n6, ns, nd, new_counter = predict_ltas_with_overlay(
                    audio, analysis_models, fft_points, canais, resample_rate, stereo_diff, retain, prev_normal, prev_6db, 
                    prev_sorted, prev_deriv, counter)
                return n1, n6, ns, nd, new_counter, n1, n6, ns, nd
            
            analyze_ltas_button.click(
                fn=ltas_analysis_wrapper,
                inputs=[audio_input, fft_n_input, canais_ltas_input, resample_ltas_input, stereo_diff_checkbox,
                       retain_ltas_checkbox, ltas_normal_state, ltas_6db_state, 
                       ltas_sorted_state, ltas_derivative_state, ltas_counter],
                outputs=[ltas_normal_output, ltas_6db_output, ltas_sorted_output, 
                        ltas_derivative_output, ltas_counter,
                        ltas_normal_state, ltas_6db_state, ltas_sorted_state, ltas_derivative_state]
            )
            
            clear_ltas_button.click(
                fn=lambda: (None, None, None, None, 0, None, None, None, None),
                outputs=[ltas_normal_output, ltas_6db_output, ltas_sorted_output, 
                        ltas_derivative_output, ltas_counter,
                        ltas_normal_state, ltas_6db_state, ltas_sorted_state, ltas_derivative_state]
            )
        #demo.launch(server_name='10.61.242.231', server_port=1111,inbrowser=True)
        # demo.queue().launch(server_name="10.61.242.231", server_port=5555, share=True, show_api=False)
        demo.queue()  # mantém a fila
        demo.launch(
            server_name="0.0.0.0",
            server_port=5555,
            share=False,             # evita limite de rede e "network error"
            show_api=False,
            inbrowser=False,
            max_threads=10,           # controla concorrência de execuções
            
        )

2025-10-06 14:05:16,107 - INFO - ✅ Configurações de ambiente carregadas
2025-10-06 14:05:16,116 - INFO - 🚀 Carregando analisador de áudio forense...
2025-10-06 14:05:16,117 - INFO - ✅ AudioForensicsAnalyzer inicializado
2025-10-06 14:05:16,118 - INFO - ✅ Analisador de áudio forense carregado com sucesso!
2025-10-06 14:05:16,119 - INFO - ✅ Analisador carregado. Iniciando Gradio...
2025-10-06 14:05:16,686 - INFO - HTTP Request: GET http://localhost:5555/gradio_api/startup-events "HTTP/1.1 200 OK"
2025-10-06 14:05:16,708 - INFO - HTTP Request: HEAD http://localhost:5555/ "HTTP/1.1 200 OK"


* Running on local URL:  http://0.0.0.0:5555

To create a public link, set `share=True` in `launch()`.


2025-10-06 15:44:00,822 - WARNING - Skipping data after last boundary
2025-10-06 15:44:00,826 - WARNING - Skipping data after last boundary
Traceback (most recent call last):
  File "/home/sepael-linux-ia/miniconda3/envs/SpeechBrain1_0/lib/python3.11/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sepael-linux-ia/miniconda3/envs/SpeechBrain1_0/lib/python3.11/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sepael-linux-ia/miniconda3/envs/SpeechBrain1_0/lib/python3.11/site-packages/gradio/blocks.py", line 2132, in process_api
    inputs = await self.preprocess_data(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/sepael-linux-ia/miniconda3/envs/SpeechBrain1_0/lib/python3.11/site-packages/gradio/blocks.py", line 1813, in pr

In [11]:
demo.close()

Closing server running on port: 5555


In [5]:
import gradio
print(gradio.__version__)

5.25.2
